# Semana 7 · Práctica guiada de OpenCV
## CADI Percepción Computacional · Experiencia IMAGEN

**Joan Schneider Beltrán Delgado** · Universidad de Cundinamarca

Esta práctica recorre las operaciones básicas de OpenCV sobre dos imágenes de
distinta naturaleza: una radiografía de tórax, con transiciones suaves, y la
fotografía de un monitor de signos vitales, con bordes marcados. La comparación
entre ambas es el eje del trabajo: el mismo código da resultados muy distintos
según qué contenga la imagen.

## 0. Preparación del entorno

In [ ]:
!pip -q install opencv-python-headless

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

## 1. Funciones de apoyo

OpenCV carga en **BGR** y Matplotlib espera **RGB**, así que para visualizar
hay que reordenar los canales.

In [ ]:
def mostrar(imagen, titulo="Imagen", cmap=None):
    """Muestra una imagen de OpenCV con los colores correctos."""
    plt.figure(figsize=(7, 5))
    if imagen.ndim == 2:
        plt.imshow(imagen, cmap=cmap or "gray")
    else:
        plt.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    plt.title(titulo)
    plt.axis("off")
    plt.show()


def nitidez(bgr):
    """Varianza del laplaciano: a mayor valor, más detalle fino conserva."""
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY) if bgr.ndim == 3 else bgr
    return round(float(cv2.Laplacian(g, cv2.CV_64F).var()), 1)

## 2. Carga e inspección

En Colab, sube la imagen con el panel de archivos o con `files.upload()`.

In [ ]:
try:
    from google.colab import files
    print("Sube la radiografía (o cualquier imagen).")
    subidos = files.upload()
    RUTA = list(subidos)[0] if subidos else "imagen_practica.jpg"
except Exception:
    RUTA = "imagen_practica.jpg"

img = cv2.imread(RUTA)
if img is None:
    raise ValueError(f"No se pudo cargar: {RUTA}")

alto, ancho = img.shape[:2]
print("Forma (alto, ancho, canales):", img.shape)
print("Tipo de dato:", img.dtype)
print("Mínimo:", img.min(), "| Máximo:", img.max())
print("Píxel central (BGR):", img[alto // 2, ancho // 2])
print("Total de píxeles:", alto * ancho)
mostrar(img, "Imagen cargada")

La forma `(alto, ancho, 3)` significa que la imagen es una matriz de
alto × ancho posiciones y que de cada una se guardan tres valores, uno por
canal de color. El tipo `uint8` limita cada valor al rango 0–255.

## 3. Conversión de color

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

print("Media por canal — B:", round(img[:, :, 0].mean(), 1),
      "| G:", round(img[:, :, 1].mean(), 1),
      "| R:", round(img[:, :, 2].mean(), 1))
print("Bytes en color:", img.nbytes, "→ en gris:", gris.nbytes)
print("Reducción:", round(100 * (1 - gris.nbytes / img.nbytes), 1), "%")

plt.figure(figsize=(14, 4))
for i, (im, t, cm) in enumerate([(img, "BGR visto directo", None),
                                 (img_rgb, "Convertida a RGB", None),
                                 (gris, "Escala de grises", "gray")], 1):
    plt.subplot(1, 3, i)
    plt.imshow(im, cmap=cm)
    plt.title(t)
    plt.axis("off")
plt.show()

En la radiografía las medias de los tres canales salen casi iguales.
Eso indica que es una imagen en escala de grises guardada en formato de color,
así que convertirla a un solo canal ahorra dos tercios de memoria sin perder
información.

## 4. Redimensionamiento y recorte

In [ ]:
redimensionada = cv2.resize(img, (320, 210))

y1, y2 = int(alto * 0.25), int(alto * 0.75)
x1, x2 = int(ancho * 0.25), int(ancho * 0.75)
recorte = img[y1:y2, x1:x2]          # seccionamiento de NumPy, no de OpenCV

print("Original:", img.shape, "→", alto * ancho, "px")
print("Redimensionada:", redimensionada.shape, "→", 320 * 210, "px")
print("Recorte:", recorte.shape)
mostrar(redimensionada, "Redimensionada")
mostrar(recorte, "Recorte central")

## 5. Filtrado y reducción de ruido

In [ ]:
blur_promedio = cv2.blur(img, (7, 7))
blur_gaussiano = cv2.GaussianBlur(img, (7, 7), 0)
blur_mediana = cv2.medianBlur(img, 7)

for nombre, im in [("Sin filtro", img), ("Promedio", blur_promedio),
                   ("Gaussiano", blur_gaussiano), ("Mediana", blur_mediana)]:
    print(f"{nombre:>10}: nitidez = {nitidez(im)}")

plt.figure(figsize=(13, 8))
for i, (im, t) in enumerate([(img, "Original"), (blur_promedio, "Promedio"),
                             (blur_gaussiano, "Gaussiano"),
                             (blur_mediana, "Mediana")], 1):
    plt.subplot(2, 2, i)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(t)
    plt.axis("off")
plt.show()

El filtro de mediana es el que menos detalle sacrifica porque
reemplaza cada píxel por el valor central de su vecindario en vez de
promediarlo, y eso respeta mejor los bordes.

## 6. Umbralización y detección de bordes

In [ ]:
gris_suave = cv2.GaussianBlur(gris, (5, 5), 0)
_, umbral = cv2.threshold(gris_suave, 127, 255, cv2.THRESH_BINARY)
v_otsu, umbral_otsu = cv2.threshold(gris_suave, 0, 255,
                                    cv2.THRESH_BINARY + cv2.THRESH_OTSU)
bordes = cv2.Canny(gris_suave, 80, 160)

print("Umbral fijo: 127 | Umbral de Otsu:", int(v_otsu))
print("Píxeles de borde:", int((bordes == 255).sum()),
      f"({round(100 * (bordes == 255).mean(), 2)} % de la imagen)")

plt.figure(figsize=(14, 4))
for i, (im, t) in enumerate([(gris, "Gris"), (umbral_otsu, "Otsu"),
                             (bordes, "Bordes (Canny)")], 1):
    plt.subplot(1, 3, i)
    plt.imshow(im, cmap="gray")
    plt.title(t)
    plt.axis("off")
plt.show()

## 7. Contornos: el problema que apareció

Siguiendo el flujo del notebook guiado, los contornos se buscan sobre el
resultado de Canny y se filtran por área. En esta imagen el resultado es
llamativo.

In [ ]:
contornos, _ = cv2.findContours(bordes, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
areas = [cv2.contourArea(c) for c in contornos]
filtrados = [a for a in areas if a > 100]

print("Contornos detectados:", len(contornos))
print("Con área > 100:", len(filtrados))
print("Área del mayor:", round(max(areas), 1) if areas else 0, "px²")

Ninguno pasa el filtro. La razón es que **lo que devuelve Canny son
líneas abiertas, no regiones cerradas**, y el área de una línea es casi cero.
El filtro no estaba mal calibrado: medía algo que no aplica.

Bajar los umbrales de Canny no resuelve nada, como muestra el barrido.

In [ ]:
for lo, hi in [(80, 160), (50, 120), (30, 90), (20, 60), (10, 40)]:
    b = cv2.Canny(gris_suave, lo, hi)
    cs, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    f = sum(1 for c in cs if cv2.contourArea(c) > 100)
    print(f"{lo:>3}/{hi:<3} → {int((b == 255).sum()):>6} px de borde, "
          f"{len(cs):>4} contornos, {f:>2} con área > 100")

La solución es buscar sobre la **máscara de Otsu**, que sí produce
regiones cerradas. Y hay que invertirla: en una radiografía la región de
interés (los pulmones) es la zona oscura.

In [ ]:
otsu_invertida = cv2.bitwise_not(umbral_otsu)

def regiones(mascara, grosor=3):
    cs, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_SIMPLE)
    lienzo = img.copy()
    encontradas = []
    for c in sorted(cs, key=cv2.contourArea, reverse=True):
        area = cv2.contourArea(c)
        if area > 100:
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(lienzo, (x, y), (x + w, y + h), (0, 255, 0), grosor)
            encontradas.append((area, w, h))
    return encontradas, lienzo

reg, resultado = regiones(otsu_invertida)
print("Regiones con área > 100:", len(reg))
for area, w, h in reg[:3]:
    print(f"   área = {int(area):>7} px²   caja = {w}x{h}")

mostrar(resultado, "Regiones detectadas sobre la máscara invertida")

Las dos regiones mayores corresponden a los campos pulmonares.

## 8. Mini reto: el mismo flujo sobre otra imagen

Se repite todo el recorrido con una imagen de naturaleza opuesta: la
fotografía de un monitor de signos vitales, con texto, líneas y recuadros de
bordes marcados. **No se cambia ningún parámetro.**

In [ ]:
RUTA2 = "imagen_reto.jpg"
img2 = cv2.imread(RUTA2)
alto2, ancho2 = img2.shape[:2]

gris2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
gris2_suave = cv2.GaussianBlur(gris2, (5, 5), 0)
mediana2 = cv2.medianBlur(img2, 7)
v2, otsu2 = cv2.threshold(gris2_suave, 0, 255,
                          cv2.THRESH_BINARY + cv2.THRESH_OTSU)
bordes2 = cv2.Canny(gris2_suave, 80, 160)

cont2, _ = cv2.findContours(bordes2, cv2.RETR_EXTERNAL,
                            cv2.CHAIN_APPROX_SIMPLE)
reto = img2.copy()
n_filtrados = 0
for c in cont2:
    if cv2.contourArea(c) > 100:
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(reto, (x, y), (x + w, y + h), (0, 255, 0), 2)
        n_filtrados += 1

print("Forma:", img2.shape, "| tipo:", img2.dtype)
print("Media por canal — B:", round(img2[:, :, 0].mean(), 1),
      "| G:", round(img2[:, :, 1].mean(), 1),
      "| R:", round(img2[:, :, 2].mean(), 1))
print("Nitidez:", nitidez(img2), "(radiografía:", nitidez(img), ")")
print("Píxeles de borde:", int((bordes2 == 255).sum()),
      f"({round(100 * (bordes2 == 255).mean(), 2)} %)")
print("Contornos:", len(cont2), "| con área > 100:", n_filtrados)

mostrar(reto, f"Mini reto: {n_filtrados} regiones detectadas")

### Qué operación fue más útil

En la **radiografía**, la umbralización de Otsu con la máscara invertida: las
transiciones son suaves y Canny apenas encuentra bordes, mientras que separar
por intensidad aísla los pulmones directamente.

En el **monitor**, al contrario, Canny funciona mejor: los bordes delimitan
cada recuadro y cada bloque de texto, que es lo que habría que aislar para
leer los valores de la pantalla.

La conclusión es que no hay una operación mejor en abstracto. El mismo código,
con los mismos números, encuentra decenas de regiones en una imagen y ninguna
en la otra. Lo que cambia no es el código sino el tipo de imagen.

## 9. Guardar los resultados

In [ ]:
carpeta = Path("resultados_opencv")
carpeta.mkdir(exist_ok=True)

for nombre, imagen in {
    "01_original.png": img,
    "02_gris.png": gris,
    "03_redimensionada.png": redimensionada,
    "04_recorte.png": recorte,
    "05_blur_mediana.png": blur_mediana,
    "06_umbral_otsu.png": umbral_otsu,
    "07_bordes_canny.png": bordes,
    "08_regiones.png": resultado,
    "09_reto_original.png": img2,
    "10_reto_bordes.png": bordes2,
    "11_reto_contornos.png": reto,
}.items():
    cv2.imwrite(str(carpeta / nombre), imagen)

print("Guardado en:", carpeta.resolve())
print(sorted(p.name for p in carpeta.iterdir()))